# Active Inference for Supply Chain Management
#
# ## Architecture
#
# **Two-model design** (generative process + preference model, fully decoupled):
#
# 1. **Generative World Model (Step Function)** — learns state transitions
#    `P(s_{t+1} | s_t, a_t)` with NO reward signal. Purely descriptive: given
#    this state and this action, what's the likely next state? This is our
#    **simulator** — it lets us roll forward beyond the actual data.
#
# 2. **Observation Model** — the existing `SupplyChainMLP` from `train.py`.
#    Maps state features → disruption predictions (likelihood, delay prob,
#    risk class, delivery deviation). Used to predict what we'd *observe*
#    given a hidden state.
#
# 3. **Homeostatic Preference Model** — specifies a target distribution the
#    system should maintain. Not reward maximization — the system maintains
#    itself within operational bounds (can't run out of money, can't stockout,
#    costs stay bounded). The preference is: over any given window, you stay
#    within x bounds.
#
# ## Step Function → MCMC-style Chaining
#
# The step function takes **current state + action** as input and returns a
# **probability distribution over next states**. We chain successive step
# functions like MCMC to simulate the network over time.
#
# - Single warehouse: one step function, straightforward
# - Network: each warehouse is a node, compose step functions, coupling =
#   transfer flows between nodes
#
# ## Actions
# - `reorder` — replenish inventory from supplier
# - `hold` — maintain current state, no intervention
# - `reroute` — redirect shipments via alternative routes
# - `expedite` — fast-track current shipments (higher cost)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import sys

# Add project src to path so we can import the existing MLP
sys.path.insert(0, str(Path.cwd() / 'src'))

DATA_PATH = Path.home() / '.cache/kagglehub/datasets/datasetengineer/logistics-and-supply-chain-dataset/versions/1/dynamic_supply_chain_logistics_dataset.csv'
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# Encode risk classification
RISK_LABELS = {"Low Risk": 0, "Moderate Risk": 1, "High Risk": 2}
df['risk_classification_enc'] = df['risk_classification'].map(RISK_LABELS)

print(f'Shape: {df.shape}')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Hourly records: {len(df)} ({len(df)/24:.0f} days, {len(df)/24/365:.1f} years)')
df.head(3)

## 1. Data Exploration & Feature Roles

Each row is one hourly observation of the logistics network. No shipment ID —
rows are connected by timestamp only. We treat this as a single time series.

In [ ]:
# === Feature roles for the POMDP ===
#
# STATE features: intrinsic system quantities the step function operates on
# These are what the world model predicts forward
state_cols = [
    'warehouse_inventory_level',  # current stock
    'order_fulfillment_status',   # how well orders are being met
    'historical_demand',          # demand signal
    'lead_time_days',             # supplier lead time
    'shipping_costs',             # current logistics cost
]

# OBSERVATION features: noisy sensor readings (partial observability)
# These are what we actually see — the world model + obs model map states to these
obs_cols = [
    'traffic_congestion_level',
    'weather_condition_severity',
    'port_congestion_level',
    'eta_variation_hours',
    'supplier_reliability_score',
    'cargo_condition_status',
    'route_risk_level',
    'customs_clearance_time',
]

# DISRUPTION targets: what the existing MLP predicts (observation model)
target_cols = [
    'disruption_likelihood_score',
    'delay_probability',
    'risk_classification_enc',
    'delivery_time_deviation',
]

# Full feature vector for the world model step function
feature_cols = state_cols + obs_cols

print('=== State features (step function input/output) ===')
print(df[state_cols].describe().round(2))
print()
print('=== Observation features (sensor readings) ===')
print(df[obs_cols].describe().round(2))
print()
print('=== Risk classification distribution ===')
print(df['risk_classification'].value_counts())

In [ ]:
# Visualize key state variables over time
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Key State Variables Over Time', fontsize=14)

plot_cols = state_cols + ['disruption_likelihood_score']
for ax, col in zip(axes.flat, plot_cols):
    # Downsample for plotting
    sample = df[['timestamp', col]].set_index('timestamp').resample('D').mean()
    ax.plot(sample.index, sample[col], linewidth=0.7)
    ax.set_title(col.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=30)

axes.flat[-1].set_visible(len(plot_cols) == 6)
plt.tight_layout()
plt.show()

## 2. Normalize & Define Action Space

**Actions**: reorder (0), hold (1), reroute (2), expedite (3)

Since the dataset has no explicit actions, we infer them from state transitions.
This gives us the **behavioral policy** — not necessarily optimal, just what
historically happened. The world model learns from these transitions regardless
of whether they were good decisions.

In [ ]:
# === Action space ===
ACTIONS = ['reorder', 'hold', 'reroute', 'expedite']
N_ACTIONS = len(ACTIONS)

# === Normalize features to [0, 1] ===
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_norm = pd.DataFrame(
    scaler.fit_transform(df[feature_cols]),
    columns=feature_cols
)

N_FEATURES = len(feature_cols)
print(f'State dim: {len(state_cols)}, Obs dim: {len(obs_cols)}, Total: {N_FEATURES}')
print(f'Actions: {ACTIONS}')
df_norm.describe().round(3)

In [ ]:
# === Infer actions from state transitions ===
# The behavioral policy — what decisions were historically made.
# We DON'T assume this is optimal. We just need the (s, a, s') tuples
# to train the world model.

def infer_action(row, next_row):
    """Infer which action was likely taken based on state change."""
    inv_change = next_row['warehouse_inventory_level'] - row['warehouse_inventory_level']
    eta_change = next_row['eta_variation_hours'] - row['eta_variation_hours']
    cost_change = next_row['shipping_costs'] - row['shipping_costs']
    disruption = row.get('disruption_likelihood_score', 0.5)

    if inv_change > 0.3:
        return 0  # reorder — large inventory increase
    elif disruption > 0.7 and eta_change < -0.1:
        return 2  # reroute — high disruption + ETA improvement
    elif disruption > 0.7 and cost_change > 0.1:
        return 3  # expedite — high disruption + cost increase
    else:
        return 1  # hold — no major intervention

# We need disruption scores for action inference, so temporarily add them
df_with_disruption = df_norm.copy()
df_with_disruption['disruption_likelihood_score'] = MinMaxScaler().fit_transform(
    df[['disruption_likelihood_score']]
)

actions = []
for i in range(len(df_with_disruption) - 1):
    a = infer_action(df_with_disruption.iloc[i], df_with_disruption.iloc[i + 1])
    actions.append(a)
actions.append(1)  # last row: hold

df_norm['action'] = actions

print('Inferred action distribution (behavioral policy):')
for i, name in enumerate(ACTIONS):
    count = (df_norm['action'] == i).sum()
    print(f'  {name}: {count:6d} ({count/len(df_norm)*100:.1f}%)')

## 3. Generative World Model — The Step Function
    
The step function is the core of the generative process:

```
step(s_t, a_t) → P(s_{t+1})
```

It takes **current state + action** as input and returns a **probability
distribution over next states** (Gaussian: mean + variance).

We chain these step functions like MCMC to simulate forward:

```
s_0 → step(s_0, a_0) → s_1 → step(s_1, a_1) → s_2 → ...
```

This gives us a **simulator** — we can roll out trajectories beyond the
actual data, sample multiple futures, and compute things like stock-out
probability (fraction of sampled trajectories where inventory hits zero).

**No reward signal.** The step function just learns the physics of the
supply chain. Actions cause state transitions, states generate observations.
We fit the dynamics, not the policy.

In [ ]:
# === Transition dataset: (s_t, a_t) → s_{t+1} ===

class TransitionDataset(Dataset):
    """Pairs of (state, action) → next_state from the time series."""

    def __init__(self, features_df, feature_cols):
        states = features_df[feature_cols].values
        actions = features_df['action'].values

        self.s_t = torch.tensor(states[:-1], dtype=torch.float32)
        self.a_t = torch.tensor(actions[:-1], dtype=torch.long)
        self.s_next = torch.tensor(states[1:], dtype=torch.float32)

    def __len__(self):
        return len(self.s_t)

    def __getitem__(self, idx):
        return self.s_t[idx], self.a_t[idx], self.s_next[idx]


# === The Step Function (World Model) ===

class StepFunction(nn.Module):
    """step(s_t, a_t) → P(s_{t+1})

    Neural net that predicts next-state distribution (Gaussian).
    Output: mean and log-variance for each state dimension.

    Chain these to simulate forward trajectories:
        s_0 → step(s_0, a_0) → s_1 → step(s_1, a_1) → ...
    """

    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super().__init__()
        self.state_dim = state_dim
        self.action_embed = nn.Embedding(n_actions, 16)

        self.net = nn.Sequential(
            nn.Linear(state_dim + 16, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, state_dim)
        self.logvar_head = nn.Linear(hidden_dim, state_dim)

    def forward(self, state, action):
        a_emb = self.action_embed(action)
        x = torch.cat([state, a_emb], dim=-1)
        h = self.net(x)
        return self.mean_head(h), self.logvar_head(h)

    def step(self, state, action):
        """Single step: returns (mean, std) of next state distribution."""
        with torch.no_grad():
            mean, logvar = self.forward(state, action)
            std = torch.exp(0.5 * logvar)
        return mean, std

    def sample_step(self, state, action):
        """Single step with stochastic sampling (for MCMC-style rollouts)."""
        mean, std = self.step(state, action)
        return mean + std * torch.randn_like(std), mean, std


def gaussian_nll(pred_mean, pred_logvar, target):
    """Negative log-likelihood for Gaussian output."""
    var = torch.exp(pred_logvar)
    return 0.5 * (pred_logvar + (target - pred_mean) ** 2 / var).mean()


print(f'StepFunction: {N_FEATURES} state dims, {N_ACTIONS} actions')

In [ ]:
# === Train/test split by time (first 2 years train, last year test) ===
split_idx = int(len(df_norm) * (2 / 3))

print(f'Train: {split_idx} rows ({df["timestamp"].iloc[0].date()} to {df["timestamp"].iloc[split_idx].date()})')
print(f'Test:  {len(df_norm) - split_idx} rows ({df["timestamp"].iloc[split_idx].date()} to {df["timestamp"].iloc[-1].date()})')

train_ds = TransitionDataset(df_norm.iloc[:split_idx], feature_cols)
test_ds = TransitionDataset(df_norm.iloc[split_idx:], feature_cols)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

In [ ]:
# === Train the step function ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

step_fn = StepFunction(N_FEATURES, N_ACTIONS).to(device)
optimizer = torch.optim.Adam(step_fn.parameters(), lr=1e-3)

N_EPOCHS = 50
train_losses, test_losses = [], []

for epoch in range(N_EPOCHS):
    step_fn.train()
    epoch_loss = 0
    for s, a, s_next in train_loader:
        s, a, s_next = s.to(device), a.to(device), s_next.to(device)
        mean, logvar = step_fn(s, a)
        loss = gaussian_nll(mean, logvar, s_next)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    step_fn.eval()
    test_loss = 0
    with torch.no_grad():
        for s, a, s_next in test_loader:
            s, a, s_next = s.to(device), a.to(device), s_next.to(device)
            mean, logvar = step_fn(s, a)
            test_loss += gaussian_nll(mean, logvar, s_next).item()
    test_losses.append(test_loss / len(test_loader))

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Train NLL: {train_losses[-1]:.4f} | Test NLL: {test_losses[-1]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(test_losses, label='Test')
plt.xlabel('Epoch'); plt.ylabel('Gaussian NLL')
plt.title('Step Function Training')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# === MCMC-style trajectory simulation ===
# Chain the step function to simulate forward. Sample stochastically
# to get a distribution of trajectories, not just a single prediction.

def simulate_trajectories(step_fn, start_state, action_policy, horizon, n_trajectories=100, device='cpu'):
    """Roll out the step function MCMC-style to generate trajectory samples.

    Args:
        step_fn: trained StepFunction
        start_state: initial state vector (1D numpy array)
        action_policy: callable(state) → action_idx, or list of actions
        horizon: number of steps to simulate
        n_trajectories: number of stochastic samples

    Returns:
        trajectories: (n_trajectories, horizon+1, state_dim) numpy array
    """
    state_dim = start_state.shape[-1]
    trajectories = np.zeros((n_trajectories, horizon + 1, state_dim))

    for traj in range(n_trajectories):
        current = torch.tensor(start_state, dtype=torch.float32, device=device).unsqueeze(0)
        trajectories[traj, 0] = start_state

        for t in range(horizon):
            # Get action
            if callable(action_policy):
                action_idx = action_policy(current.cpu().numpy()[0])
            else:
                action_idx = action_policy[t % len(action_policy)]
            action = torch.tensor([action_idx], device=device)

            # Stochastic step (sample from predicted distribution)
            sampled, mean, std = step_fn.sample_step(current, action)
            current = sampled
            trajectories[traj, t + 1] = sampled.cpu().numpy()[0]

    return trajectories


# === Simulate with behavioral policy (hold) and compare to actual ===
step_fn.eval()
start = test_ds.s_t[0].numpy()
horizon = 200

# Simulate 100 stochastic trajectories under "hold" action
hold_trajectories = simulate_trajectories(
    step_fn, start, action_policy=[1],  # hold
    horizon=horizon, n_trajectories=100, device=device
)

# Also get deterministic rollout for comparison
actual = test_ds.s_next[:horizon].numpy()

# Plot: mean trajectory ± std vs actual for key features
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle(f'MCMC Trajectory Simulation: 100 samples, {horizon}-step horizon', fontsize=14)

plot_feats = ['warehouse_inventory_level', 'shipping_costs', 'eta_variation_hours',
              'traffic_congestion_level', 'order_fulfillment_status', 'supplier_reliability_score']

for ax, feat in zip(axes.flat, plot_feats):
    idx = feature_cols.index(feat)
    traj_mean = hold_trajectories[:, 1:, idx].mean(axis=0)
    traj_std = hold_trajectories[:, 1:, idx].std(axis=0)

    ax.plot(actual[:, idx], label='Actual', linewidth=1.5, color='black', alpha=0.7)
    ax.plot(traj_mean, label='Mean prediction', linewidth=1.5, color='blue')
    ax.fill_between(range(horizon), traj_mean - 2*traj_std, traj_mean + 2*traj_std,
                    alpha=0.2, color='blue', label='±2σ')
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# === Stock-out probability ===
# "What fraction of sampled trajectories hit zero inventory?"
inv_idx = feature_cols.index('warehouse_inventory_level')
stockout_per_traj = (hold_trajectories[:, :, inv_idx] <= 0.05).any(axis=1)
print(f'\nStock-out probability (hold policy, {horizon}h): '
      f'{stockout_per_traj.mean():.1%} of trajectories hit near-zero inventory')

## 4. Homeostatic Preference Model

The preference model encodes **what homeostasis means** for this system.

Not reward maximization. The system maintains itself within operational bounds:
- You can't run out of inventory (stockout)
- You can't let costs blow up
- You need to keep fulfilling orders

The preference is: **over any given window, you have stayed within x bounds.**
Karl Friston would say you don't need a specific reward — you just want the
system at homeostasis. The preference is a prior belief over observations
(the C matrix in active inference).

Precision controls how rigid each bound is. High precision = non-negotiable.
Low precision = flexible. Ultimately the customer of the system specifies
their own target distribution.

In [ ]:
class HomeostaticPreferences:
    """Target distribution for homeostatic control.

    Each preference is a (target, precision) pair in normalized [0,1] space.
    Higher precision = less tolerance for deviation = non-negotiable bound.

    The ideal scenario: you do not get affected by disruptions.
    I.e., you minimize how much prediction error accumulates over any given window.
    """

    def __init__(self, feature_cols):
        self.feature_cols = feature_cols
        self.targets = {}

        # Inventory: moderate-high (can't stockout, can't over-accumulate)
        self.targets['warehouse_inventory_level'] = (0.6, 5.0)
        # Order fulfillment: must stay high (non-negotiable)
        self.targets['order_fulfillment_status'] = (0.85, 8.0)
        # Shipping costs: keep low (budget constraint)
        self.targets['shipping_costs'] = (0.25, 3.0)
        # ETA variation: minimize (stable delivery = homeostasis)
        self.targets['eta_variation_hours'] = (0.1, 4.0)
        # Lead time: keep reasonable
        self.targets['lead_time_days'] = (0.2, 2.0)
        # Traffic/congestion: want low (environmental factor we can partially influence)
        self.targets['traffic_congestion_level'] = (0.2, 1.5)

    def compute_deviation(self, state):
        """Precision-weighted squared deviation from homeostatic targets."""
        if isinstance(state, torch.Tensor):
            state = state.cpu().numpy()
        if state.ndim == 1:
            state = state.reshape(1, -1)

        total = np.zeros(state.shape[0])
        for feat, (target, precision) in self.targets.items():
            if feat in self.feature_cols:
                idx = self.feature_cols.index(feat)
                total += precision * (state[:, idx] - target) ** 2
        return total

    def in_bounds(self, state, tolerance=0.3):
        """Check if state is within homeostatic bounds (for recovery metric)."""
        return self.compute_deviation(state) < tolerance

    def log_preference(self, state):
        """Log-probability under preference distribution. Higher = more preferred."""
        return -self.compute_deviation(state)


prefs = HomeostaticPreferences(feature_cols)

# === How well does the historical data maintain homeostasis? ===
deviations = prefs.compute_deviation(df_norm[feature_cols].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df['timestamp'], deviations, linewidth=0.3, alpha=0.6)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Homeostatic Deviation')
axes[0].set_title('Historical Deviation Over Time (lower = better homeostasis)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(deviations, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=np.mean(deviations), color='red', linestyle='--', label=f'Mean: {np.mean(deviations):.2f}')
axes[1].set_xlabel('Deviation')
axes[1].set_title('Distribution of Homeostatic Deviation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

in_bounds_pct = prefs.in_bounds(df_norm[feature_cols].values).mean()
print(f'Mean deviation: {deviations.mean():.3f} (std: {deviations.std():.3f})')
print(f'Historical time in homeostatic bounds: {in_bounds_pct:.1%}')

## 5. Active Inference Agent

Uses the step function as a simulator to evaluate candidate action policies.

**Expected Free Energy** (following Parr, Pezzulo, & Friston 2022; Da Costa et al. 2020):

For a policy π and future time τ:

$$G(\pi, \tau) = -\underbrace{\mathbb{E}_{q(o_\tau|\pi)}[\ln \tilde{p}(o_\tau)]}_{\text{pragmatic value}} - \underbrace{\mathbb{E}_{q(s_\tau|\pi)}[H[p(o_\tau|s_\tau)]]}_{\text{epistemic value (info gain)}}$$

Where:
- $\tilde{p}(o_\tau)$ is the **prior preference** over observations (C matrix) — our
  homeostatic target distribution
- $H[p(o_\tau|s_\tau)]$ is the **ambiguity** — entropy of the observation model given
  the state. Lower ambiguity means the state is more informative about what we'll observe.

**Pragmatic value**: expected log-preference of predicted observations under policy π.
Policies that lead to preferred (homeostatic) outcomes score higher.

**Epistemic value**: expected information gain. Policies that reduce uncertainty
about hidden states score higher (drives exploration).

The agent selects the policy that **minimizes** G (equivalently, maximizes
negative G = pragmatic value + epistemic value).

In [ ]:
class ActiveInferenceAgent:
    """Selects policies by minimizing expected free energy.

    G(π) = Σ_τ G(π, τ) where:
        G(π, τ) = -E_q(o_τ|π)[ln C(o_τ)]          (pragmatic: preference)
                  -E_q(s_τ|π)[H[P(o_τ|s_τ)]]      (epistemic: ambiguity)

    In our continuous Gaussian setting:
        - Pragmatic term: -log C(o) = precision-weighted deviation from targets
          (log-preference under Gaussian prior = negative Mahalanobis distance)
        - Epistemic term: entropy of the predictive distribution H[q(s_τ|π)]
          = 0.5 * Σ_i log(2πe σ_i²). Policies that pass through high-uncertainty
          states are penalized (ambiguity), but policies that RESOLVE uncertainty
          are preferred (info gain manifests as reduction in future entropy).
    """

    def __init__(self, step_fn, preferences, n_actions, planning_horizon=5, n_rollouts=30):
        self.step_fn = step_fn
        self.preferences = preferences
        self.n_actions = n_actions
        self.planning_horizon = planning_horizon
        self.n_rollouts = n_rollouts
        self.device = next(step_fn.parameters()).device

    def expected_free_energy(self, state, policy):
        """Compute G(π) = Σ_τ G(π,τ) for a policy (sequence of actions).

        G(π,τ) = -E_q[ln C(o_τ)] + H[q(o_τ|π)]

        First term: negative log-preference (pragmatic) — how far are predicted
        observations from the homeostatic target distribution C?

        Second term: predictive entropy (ambiguity) — how uncertain is the
        step function about the next state? High entropy = ambiguous state.
        """
        current = state.clone()
        G = 0.0

        for tau in range(len(policy)):
            action = torch.tensor([policy[tau]], device=self.device)
            mean, std = self.step_fn.step(current, action)

            # --- Pragmatic term: -E_q[ln C(o_τ)] ---
            # C(o) is the prior preference (Gaussian target distribution).
            # -ln C(o) ∝ precision * (o - target)²
            # This is exactly what compute_deviation returns.
            pragmatic = self.preferences.compute_deviation(mean.cpu().numpy())[0]

            # --- Epistemic term: H[q(o_τ|π)] = ambiguity ---
            # For Gaussian: H = 0.5 * Σ_i ln(2πe σ_i²)
            # This captures how uncertain the world model is about the predicted state.
            # Actions that lead to less ambiguous states are preferred.
            ambiguity = 0.5 * torch.log(2 * np.pi * np.e * std ** 2).sum().item()

            G += pragmatic + ambiguity

            # Advance using predicted mean (planning)
            current = mean

        return G

    def select_action(self, state):
        """Evaluate all first-step actions with random policy continuations.

        Returns (best_action, G_per_action) where G_per_action[i] is the
        average expected free energy for policies starting with action i.
        """
        s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)

        G_per_action = []
        for a0 in range(self.n_actions):
            G_total = 0.0
            for _ in range(self.n_rollouts):
                policy = [a0] + list(np.random.randint(0, self.n_actions, self.planning_horizon - 1))
                G_total += self.expected_free_energy(s, policy)
            G_per_action.append(G_total / self.n_rollouts)

        # Policy selection via softmin: π* = argmin G(π)
        best = int(np.argmin(G_per_action))
        return best, G_per_action


agent = ActiveInferenceAgent(step_fn, prefs, N_ACTIONS, planning_horizon=5, n_rollouts=30)
print('Active inference agent ready.')
print(f'  Planning horizon: {agent.planning_horizon} steps')
print(f'  Rollouts per action: {agent.n_rollouts}')
print(f'  Total policy evaluations per decision: {N_ACTIONS * agent.n_rollouts}')

## 6. Evaluation: Active Inference vs Behavioral Policy

Both agents start from the same state and use the **same step function** (simulator)
to evolve forward. We compare:
- **Behavioral policy**: replays the inferred historical actions
- **Active inference**: selects actions to minimize expected free energy

Metrics: homeostatic deviation, stock-out probability, action distribution.

In [ ]:
# === Run simulation: behavioral vs active inference ===
n_eval = 200
start_state = test_ds.s_t[0].numpy()

# --- Behavioral policy (replay historical actions) ---
beh_states = [start_state.copy()]
beh_devs = [prefs.compute_deviation(start_state)[0]]
current = torch.tensor(start_state, dtype=torch.float32, device=device).unsqueeze(0)

for t in range(n_eval):
    action = test_ds.a_t[t:t+1].to(device)
    mean, std = step_fn.step(current, action)
    current = mean
    s = mean.cpu().numpy()[0]
    beh_states.append(s)
    beh_devs.append(prefs.compute_deviation(s)[0])

# --- Active inference agent ---
ai_states = [start_state.copy()]
ai_devs = [prefs.compute_deviation(start_state)[0]]
ai_actions = []
current = start_state.copy()

print(f'Running active inference agent for {n_eval} steps...')
for t in range(n_eval):
    action, G_values = agent.select_action(current)
    ai_actions.append(action)

    s_t = torch.tensor(current, dtype=torch.float32, device=device).unsqueeze(0)
    a_t = torch.tensor([action], device=device)
    mean, std = step_fn.step(s_t, a_t)
    current = mean.cpu().numpy()[0]
    ai_states.append(current)
    ai_devs.append(prefs.compute_deviation(current)[0])

    if (t + 1) % 50 == 0:
        print(f'  Step {t+1}/{n_eval} | AI dev: {ai_devs[-1]:.3f} | '
              f'Behavioral dev: {beh_devs[-1]:.3f} | Action: {ACTIONS[action]}')

print('Done.')

In [ ]:
# === Results ===
beh_arr = np.array(beh_states)
ai_arr = np.array(ai_states)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Homeostatic deviation over time
ax = axes[0, 0]
ax.plot(beh_devs, label='Behavioral', alpha=0.8)
ax.plot(ai_devs, label='Active Inference', alpha=0.8)
ax.set_xlabel('Step'); ax.set_ylabel('Deviation')
ax.set_title('Homeostatic Deviation')
ax.legend(); ax.grid(True, alpha=0.3)

# 2. Cumulative deviation
ax = axes[0, 1]
ax.plot(np.cumsum(beh_devs), label='Behavioral', alpha=0.8)
ax.plot(np.cumsum(ai_devs), label='Active Inference', alpha=0.8)
ax.set_xlabel('Step'); ax.set_ylabel('Cumulative')
ax.set_title('Cumulative Homeostatic Cost')
ax.legend(); ax.grid(True, alpha=0.3)

# 3. Action distribution
ax = axes[0, 2]
beh_counts = np.bincount(test_ds.a_t[:n_eval].numpy(), minlength=N_ACTIONS)
ai_counts = np.bincount(ai_actions, minlength=N_ACTIONS)
x = np.arange(N_ACTIONS); w = 0.35
ax.bar(x - w/2, beh_counts / n_eval, w, label='Behavioral')
ax.bar(x + w/2, ai_counts / n_eval, w, label='Active Inference')
ax.set_xticks(x); ax.set_xticklabels(ACTIONS, rotation=15)
ax.set_ylabel('Proportion'); ax.set_title('Action Distribution')
ax.legend(); ax.grid(True, alpha=0.3)

# 4. Inventory trajectory
inv_idx = feature_cols.index('warehouse_inventory_level')
ax = axes[1, 0]
ax.plot(beh_arr[:, inv_idx], label='Behavioral', alpha=0.8)
ax.plot(ai_arr[:, inv_idx], label='Active Inference', alpha=0.8)
ax.axhline(y=0.6, color='green', ls='--', alpha=0.5, label='Target')
ax.axhline(y=0.05, color='red', ls='--', alpha=0.5, label='Stockout threshold')
ax.set_xlabel('Step'); ax.set_ylabel('Inventory (norm.)')
ax.set_title('Inventory Level'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# 5. Shipping costs trajectory
cost_idx = feature_cols.index('shipping_costs')
ax = axes[1, 1]
ax.plot(beh_arr[:, cost_idx], label='Behavioral', alpha=0.8)
ax.plot(ai_arr[:, cost_idx], label='Active Inference', alpha=0.8)
ax.axhline(y=0.25, color='green', ls='--', alpha=0.5, label='Target')
ax.set_xlabel('Step'); ax.set_ylabel('Cost (norm.)')
ax.set_title('Shipping Costs'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# 6. Order fulfillment trajectory
ful_idx = feature_cols.index('order_fulfillment_status')
ax = axes[1, 2]
ax.plot(beh_arr[:, ful_idx], label='Behavioral', alpha=0.8)
ax.plot(ai_arr[:, ful_idx], label='Active Inference', alpha=0.8)
ax.axhline(y=0.85, color='green', ls='--', alpha=0.5, label='Target')
ax.set_xlabel('Step'); ax.set_ylabel('Fulfillment (norm.)')
ax.set_title('Order Fulfillment'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# === Stock-out probability comparison ===
beh_stockout = (beh_arr[:, inv_idx] <= 0.05).mean()
ai_stockout = (ai_arr[:, inv_idx] <= 0.05).mean()

# Time in homeostatic bounds
beh_in_bounds = prefs.in_bounds(beh_arr).mean()
ai_in_bounds = prefs.in_bounds(ai_arr).mean()

print(f'\n{"="*50}')
print(f'{"Metric":<35} {"Behavioral":>10} {"AI Agent":>10}')
print(f'{"="*50}')
print(f'{"Mean deviation":<35} {np.mean(beh_devs):>10.3f} {np.mean(ai_devs):>10.3f}')
print(f'{"Stock-out rate":<35} {beh_stockout:>10.1%} {ai_stockout:>10.1%}')
print(f'{"Time in homeostatic bounds":<35} {beh_in_bounds:>10.1%} {ai_in_bounds:>10.1%}')
print(f'{"Cumulative cost":<35} {np.sum(beh_devs):>10.1f} {np.sum(ai_devs):>10.1f}')
improvement = (np.mean(beh_devs) - np.mean(ai_devs)) / np.mean(beh_devs) * 100
print(f'{"Improvement":<35} {"":>10} {improvement:>9.1f}%')

## 7. Next Steps

### Immediate
- [ ] Integrate existing `SupplyChainMLP` checkpoint as observation model
      (state features → disruption predictions) alongside the step function
- [ ] Fix torch DLL issue or switch to a clean venv
- [ ] Validate step function on held-out sequences before plugging in policy

### Architecture extensions
- [ ] **Multi-warehouse network**: each warehouse = node, compose step functions,
      coupling = transfer flows between nodes. Stock-out probability = fraction of
      sampled trajectories hitting zero at each node.
- [ ] **Closed-form stock-out**: since the single-warehouse case may be linear,
      explore analytical expressions instead of Monte Carlo

### Active inference improvements
- [ ] Learn preference parameters from data (computational phenotyping —
      inverse RL to infer what ranking of outcomes explains observed behavior)
- [ ] Add proper epistemic actions (explore disruption states before committing)
- [ ] Softmax policy selection with temperature parameter instead of argmin
- [ ] Compare against: Thompson sampling, UCB, simple neural net baseline

### Data
- [ ] Infer disruption severity from state transitions (not explicitly labeled)
- [ ] Figure out row grouping (cluster by GPS coordinates to identify routes?)